[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/lithology/upper_tweed_lithology_and_topography.ipynb)

# Lithology and topography: the upper Tweed, Southern Uplands of Scotland

Does the rock type control how steep rivers and hillslopes are? This notebook takes the **upper Tweed**, from its source to just above its confluence with the Ettrick Water near Galashiels (about 980 km²), and compares its topography across the rock units of the geological map.

The workflow is:
1. Install `lsdviztools3` (plotting) and the `lsdtt3` command-line tools.
2. Download a **30 m** Copernicus DEM, make a hillshade, and get the **geological map** of the area from [Macrostrat](https://macrostrat.org) with `lsdtt-fetch-lithology`.
3. Find the **concavity index** (m/n) of the whole catchment with a **disorder scan**.
4. Compute the **channel steepness index** k<sub>sn</sub> at that concavity, and split it by **lithologic unit**.
5. Map it all: the catchment and its channels with a locator map, the geological map, and k<sub>sn</sub> in five classes.
6. Compute hillslope **gradient** and **curvature** from a polynomial surface fitted to the DEM, and plot every metric by unit.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

**`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

### Install PyGMT

`lsdviztools3` draws its maps with PyGMT, which needs the GMT library. pip cannot install GMT, so we install PyGMT (and GMT with it) from conda-forge. This is the slowest step: repeated tests suggest it takes around 70 seconds.

In [ ]:
# mamba prints hundreds of lines, so we hide them and only show the end of
# the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Install lsdviztools3 from PyPI

`lsdviztools3`, the plotting package, is on [PyPI](https://pypi.org/project/lsdviztools3/). It needs Python 3.12 or later, which Colab has.

In [ ]:
!pip install -q "lsdviztools3>=0.4.2"

### Get the lsdtt3 command-line tools

The Linux package of the latest [lsdtt3 release](https://github.com/LSDtopotools/lsdtt3/releases/latest). `releases/latest/download/` always points at the newest version.

In [ ]:
!wget -q https://github.com/LSDtopotools/lsdtt3/releases/latest/download/lsdtt3-backend-linux-x86_64-core.tar.gz
!tar -xzf lsdtt3-backend-linux-x86_64-core.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import glob
import os

root = sorted(glob.glob("/content/lsdtt3-backend-linux-x86_64-core-v*"))[-1]
print("Using", root)
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's proj.db breaks rasterio and pyproj, which lsdviztools3 uses.

In [ ]:
!lsdtt-chi-analysis -v

## Step 1: The DEM, a hillshade and the outlet

`lsdtt-fetch-raster` downloads the **Copernicus GLO-30** DEM (`dem_source: cop30_aws`, 30 m, no account or key) for a box given in latitude and longitude, and projects it to the UTM zone of the box (here zone 30N), so distances are in metres.

The box must hold the **whole catchment**: a basin that touches the edge of the DEM is incomplete, and the chi analysis leaves it out. The Tweed rises in the hills near the southern edge of this box, and its northern tributaries (the Eddleston, Leithen and Lyne Waters) reach the Moorfoot and Pentland hills.

In [ ]:
# Lower left (south-west) and upper right (north-east) corners, as copied from Google Maps
lower_left = (55.38, -3.62)
upper_right = (55.85, -2.85)

with open("tweed_fetch.param", "w") as f:
    f.write("write_fname: tweed\n")
    f.write(f"west: {lower_left[1]}\n")
    f.write(f"east: {upper_right[1]}\n")
    f.write(f"south: {lower_left[0]}\n")
    f.write(f"north: {upper_right[0]}\n")
    f.write("dem_source: cop30_aws\n")
!lsdtt-fetch-raster ./ tweed_fetch.param

with open("tweed_hillshade.param", "w") as f:
    f.write("read_fname: tweed_DEM.tif\n")
    f.write("write_fname: tweed_hs\n")
    f.write("write_output_format: gtiff\n")
    f.write("write_hillshade_raster: true\n")
!lsdtt-raster-preprocessing ./ tweed_hillshade.param

**The outlet** is on the Tweed just above where the Ettrick Water joins it. lsdtt3 reads outlet points from a CSV with `latitude` and `longitude` columns. The point does not have to be exactly on the river: `basin_outlet_snapping_method: expanding_square` moves it to the nearest channel pixel.

In [ ]:
import pandas as pd

pd.DataFrame({"latitude": [55.603905982611145], "longitude": [-2.8791451327374133]}).to_csv("tweed_outlet.csv", index=False)

## Step 2: The geological map

`lsdtt-fetch-lithology` downloads geological map units from [Macrostrat](https://macrostrat.org), which compiles geological maps from around the world. For Great Britain its map is the **British Geological Survey's 1:625 000 map** (DiGMapGB-625). The tool writes the units as polygons (`tweed_macrostrat_units.fgb`) with a field `lith_primary`: the first lithology of each map unit, such as `greywacke` or `sandstone`. It also writes `tweed_lithology_vocabulary.csv`, listing the lithologies and their areas, and `tweed_ATTRIBUTION.txt`, the citation for the maps.

**A 1:625 000 map is coarse.** Its contacts can be hundreds of metres out on the ground, and a "unit" groups rocks of one age and setting, not one rock type. Keep that in mind when a channel runs close to a contact.

In [ ]:
with open("tweed_lithology.param", "w") as f:
    f.write("write_fname: tweed\n")
    f.write(f"west: {lower_left[1]}\n")
    f.write(f"east: {upper_right[1]}\n")
    f.write(f"south: {lower_left[0]}\n")
    f.write(f"north: {upper_right[0]}\n")
!lsdtt-fetch-lithology ./ tweed_lithology.param

print(pd.read_csv("tweed_lithology_vocabulary.csv").round(1).to_string(index=False))
print()
print(open("tweed_ATTRIBUTION.txt").read())

The areas are for the whole box, and the polygons reach beyond it.

## Step 3: The concavity of the whole catchment from a disorder scan

### How it works (Mudd et al., 2018)

Chi (χ) is a coordinate along each channel that integrates drainage area upstream from the outlet:

```
χ = ∫ (A₀ / A(x))^(m/n) dx
```

where A is drainage area, A₀ a reference area (1 m²) and **m/n the concavity index**. If a basin is in steady state, with uniform uplift and erodibility, and m/n is right, **every channel in the basin collapses onto one line** in a plot of elevation against chi.

The **disorder** measures how far the channels are from collapsing onto one curve. Sort all the channel pixels of the basin by elevation: if chi only ever increases with elevation, the channels are perfectly collapsed, and every reversal adds to the disorder. lsdtt3 computes the disorder for each m/n in a range, and the best m/n is the one with the least.

With `run_bootstrap: true` the tool also resamples the basin's side branches (keeping the main stem) `bootstrap_replicates` times and recomputes the disorder at every m/n. This gives a 95% interval for each m/n: an m/n whose interval overlaps that of the best one **cannot be told apart from it**.

### The settings

* `basin_outlet_fname`: the outlet from Step 1. We analyse **one basin, the whole catchment**.
* `threshold_contributing_pixels: 2222`: a channel drains at least 2222 pixels of 30 m × 30 m, which is **2 km²**.
* `mn_min`, `mn_max`, `mn_step`: the m/n values to test.

The settings shared with the k<sub>sn</sub> run in Step 4 go in one dictionary.

In [ ]:
common = {
    "read_fname": "tweed_DEM.tif",
    "basin_outlet_fname": "tweed_outlet.csv",
    "basin_outlet_snapping_method": "expanding_square",
    "threshold_contributing_pixels": 2222,                  # 2 km2 of 30 m pixels
    "basin_outlet_contributing_pixels_threshold": 2222,
    "n_largest_basins": 0,
    "fit_chi_networks": "true",
    "fit_critical_divergence_metres": 10,                   # see Step 4
}


def write_param(fname, settings):
    """Write a dictionary of settings as an lsdtt3 parameter file."""
    with open(fname, "w") as f:
        for key, value in settings.items():
            f.write(f"{key}: {value}\n")


sweep_settings = dict(common)
sweep_settings.update({
    "write_fname": "tweed_disorder",
    "mn_min": 0.1,
    "mn_max": 0.9,
    "mn_step": 0.05,
    "run_bootstrap": "true",
    "bootstrap_replicates": 100,
    # Only the disorder is needed from this run, not the per-m/n fit tables
    "write_fit_tables": "false",
    "write_chi_tributary_table": "false",
    "write_chi_knickpoint_table": "false",
    "write_ksn_basin_summary": "false",
})
write_param("tweed_disorder.param", sweep_settings)
!lsdtt-chi-analysis ./ tweed_disorder.param

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sweep = pd.read_csv("tweed_disorder_chi_disorder_sweep.csv")
sweep["m_over_n"] = sweep["m_over_n"].round(3)
best = sweep.loc[sweep["full_network_disorder"].idxmin()]
best_mn = float(best["m_over_n"])
same = sweep[(sweep["bootstrap_percentile_2p5"] <= best["bootstrap_percentile_97p5"]) &
             (sweep["bootstrap_percentile_97p5"] >= best["bootstrap_percentile_2p5"])]
print(f"{int(best['n_side_branches'])} side branches. Least disorder at m/n = {best_mn}; "
      f"m/n from {same['m_over_n'].min()} to {same['m_over_n'].max()} cannot be told apart from it")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sweep["m_over_n"], sweep["full_network_disorder"], "o-", color="black", label="whole network")
ax.fill_between(sweep["m_over_n"], sweep["bootstrap_percentile_2p5"], sweep["bootstrap_percentile_97p5"],
                color="tab:blue", alpha=0.3, label="bootstrap 95% interval")
ax.plot(sweep["m_over_n"], sweep["bootstrap_mean"], color="tab:blue", label="bootstrap mean")
ax.axvline(best_mn, color="k", ls="--", lw=1)
ax.axvspan(same["m_over_n"].min(), same["m_over_n"].max(), color="0.85", zorder=0, label="indistinguishable from the best")
ax.set_xlabel("m/n")
ax.set_ylabel("Disorder")
ax.set_title("Disorder of the upper Tweed")
ax.legend()
plt.show()

**How to read the bootstrap.** A resample draws side branches with replacement, so it has fewer distinct branches, and a branch drawn twice collapses onto its copy. The bootstrap disorder is therefore lower than the whole-network disorder: compare the bootstrap intervals with one another, not with the black line.

We use the m/n with the least disorder from here on.

**Questions:**

* How wide is the range of m/n that cannot be told apart from the best? Does it reach either end of the range tested?
* The disorder method assumes **uniform erodibility and uplift** and a network near steady state. How well might those assumptions hold in a catchment that crosses several rock units?

## Step 4: k<sub>sn</sub>, split by lithology

Now we run `lsdtt-chi-analysis` again at the chosen m/n only (`m_over_n`). It fits each channel with straight segments in chi–elevation space: the gradient of each segment is its **normalised channel steepness index** k<sub>sn</sub> (with A₀ = 1 m², in units of m<sup>2m/n</sup>). A segment ends where the data move more than `fit_critical_divergence_metres` (10 m, which suits a 30 m DEM) from the line.

With `lithology_fname` the run also splits the network by rock unit: every channel node takes the unit of the pixel it sits on, and k<sub>sn</sub> is summarised per unit. We give it

* `lithology_fname`: the Macrostrat polygons from Step 2, with `lithology_field_name: lith_primary`. Each distinct value is a unit, numbered 1, 2, ... in this run (`tweed_ksn_lithology_units.csv` is the legend);
* `write_lithology_code_raster: true`: the units as a raster on the DEM grid (`tweed_ksn_lithology_code.tif`), which we use for the map and for the hillslope statistics in Step 6.

It also writes the channel network as lines (`write_chi_data_map_lines`) and the fitted segments (`write_chi_fit_segment_lines`), for the maps.

In [ ]:
mn_suffix = f"{best_mn:.3f}".replace(".", "p")   # how lsdtt3 writes m/n in file names, e.g. 0p650

ksn_settings = dict(common)
ksn_settings.update({
    "write_fname": "tweed_ksn",
    "m_over_n": best_mn,
    "write_chi_data_maps": "true",
    "chi_data_maps_write_vector_format": "flatgeobuf",
    "write_chi_data_map_lines": "true",
    "write_chi_fit_segment_lines": "true",
    "write_vector_format": "flatgeobuf",
    "lithology_fname": "tweed_macrostrat_units.fgb",
    "lithology_field_name": "lith_primary",
    "write_lithology_code_raster": "true",
})
write_param("tweed_ksn.param", ksn_settings)
!lsdtt-chi-analysis ./ tweed_ksn.param > tweed_ksn.log
!grep -A 12 "k_sn by lithologic unit" tweed_ksn.log

The table printed at the end of the run is the core result. For each unit:

* the **reach median k<sub>sn</sub>**: a *lithologic reach* is a run of channel nodes in one unit; its k<sub>sn</sub> is the slope of elevation against chi over that run. The interval in brackets is a bootstrap over reaches;
* the number of reaches (and how many are long enough to use);
* the **node median k<sub>sn</sub>**: the median over every channel node in the unit, each carrying the k<sub>sn</sub> of its fitted segment;
* the numbers of nodes, segments and basins.

A unit with fewer than 5 usable reaches gets no interval. Its numbers rest on a few short stretches of channel.

In [ ]:
by_unit = pd.read_csv(f"tweed_ksn_chi_ksn_by_lithology_mn_{mn_suffix}.csv")
by_unit[["lith_name", "n_nodes", "channel_length_m", "k_sn_median", "k_sn_q25", "k_sn_q75",
         "n_reaches_qualifying", "reach_k_sn_median", "low_sample"]].round(1)

**One colour per unit.** The unit codes are numbered within each run, so the plotting tools match colours by unit **name**. We fix a colour for each name and save them as `tweed_unit_colours.csv` (columns `name,colour`), which the map and every chart below use. A name not in the list gets a colour from a standard palette.

In [ ]:
from matplotlib.colors import to_hex

units = pd.read_csv("tweed_ksn_lithology_units.csv")
chosen = {"greywacke": "#7f9c5a", "sandstone": "#e6b85c", "tuff": "#d36fb4", "mudstone": "#9c6b4e",
          "claystone": "#6a8fc2", "sedimentary": "#c9c29a", "igneous": "#d7301f", "unmapped": "#ffffff"}
spare = iter(to_hex(c) for c in plt.cm.tab10.colors)
colours = {name: chosen.get(name) or next(spare) for name in units.name.fillna("unmapped")}
pd.DataFrame({"name": list(colours), "colour": list(colours.values())}).to_csv("tweed_unit_colours.csv", index=False)
colours

## Step 5: Maps

### The catchment, its channels and where it is

`lsdviz3-channels` draws the channel network (2 km² and larger) over the hillshade. `--locator regional` adds a small map of the region, with the DEM's footprint in red.

In [ ]:
from IPython.display import Image, display

!lsdviz3-channels --base tweed_hs_hillshade.tif --lines tweed_ksn_chi_data_map_lines.fgb \
    --line-color blue --locator regional --locator-position BR --out tweed_network_map.png
display(Image("tweed_network_map.png", width=700))

### The geological map

`lsdviz3-lithology-map` draws the unit raster over the hillshade, with a legend of unit names and the credit line for the map. `--channels` adds the channel nodes of the k<sub>sn</sub> run.

In [ ]:
!lsdviz3-lithology-map --codes tweed_ksn_lithology_code.tif --dem tweed_DEM.tif \
    --channels tweed_ksn_chi_ksn_lithology_nodes_mn_{mn_suffix}.csv --unit-colours tweed_unit_colours.csv \
    --opaque --out tweed_lithology_map.png
display(Image("tweed_lithology_map.png", width=800))

**Questions:** Which units do the channels cross, and where in the catchment is each one?

### k<sub>sn</sub> in five classes

`lsdviz3-chi` draws the fitted network coloured by k<sub>sn</sub>. `--ksn-classes natural_breaks --n-classes 5` puts k<sub>sn</sub> into **five classes**, with breaks found in the data (Jenks natural breaks, in log10 k<sub>sn</sub>). `--no-stacked` skips the stacked chi plot.

In [ ]:
!lsdviz3-chi --data-dir . --prefix tweed_ksn --out-prefix tweed_classes --m-over-n {best_mn} \
    --dem tweed_DEM.tif --out-dir . --no-stacked --ksn-classes natural_breaks --n-classes 5
display(Image("tweed_classes_ksn_map.png", width=800))

## Step 6: Landscape metrics by lithology

### k<sub>sn</sub>

`lsdviz3-ksn-by-lithology` reads the per-unit tables of the k<sub>sn</sub> run and draws box plots and histograms of k<sub>sn</sub> in each unit. Units with few usable reaches are hatched.

In [ ]:
!lsdviz3-ksn-by-lithology --data-dir . --prefix tweed_ksn --m-over-n {best_mn} --out-dir . \
    --unit-colours tweed_unit_colours.csv
for png in ["tweed_ksn_ksn_box.png", "tweed_ksn_reach_ksn_box.png", "tweed_ksn_ksn_hist_length.png"]:
    display(Image(png, width=800))

### Hillslope gradient and curvature from a fitted surface

`lsdtt-surface-metrics` fits a quadratic surface to the DEM around every pixel, within `polyfit_lambda_metres` (90 m, three pixels), and computes from it

* `slope`: the **gradient** of the surface (m/m; 0.1 is a 10% slope);
* `mean_curvature`: how convex or concave the surface is (1/m). On hillslopes, ridges and spurs are convex and hollows and valleys concave.

We then compare them, and the elevation, across the units **inside the catchment**: we cut the rasters with the catchment polygon of the chi run, and group the pixels by the unit raster of Step 4. These are all pixels, hillslopes and valley floors together.

In [ ]:
with open("tweed_polyfit.param", "w") as f:
    f.write("read_fname: tweed_DEM.tif\n")
    f.write("write_fname: tweed\n")
    f.write("polyfit_metrics: slope,mean_curvature\n")
    f.write("polyfit_lambda_metres: 90\n")
!lsdtt-surface-metrics ./ tweed_polyfit.param

In [ ]:
import geopandas as gpd
import rasterio
from rasterio import features


def read_raster(fname):
    with rasterio.open(fname) as src:
        data = src.read(1).astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
    return data


units = pd.read_csv("tweed_ksn_lithology_units.csv")
unit_name = dict(zip(units.code, units.name))
codes = read_raster("tweed_ksn_lithology_code.tif")

catchment = gpd.read_file("tweed_ksn_chi_basin_polygons.fgb")
with rasterio.open("tweed_ksn_lithology_code.tif") as src:
    inside = features.rasterize([(g, 1) for g in catchment.geometry], out_shape=src.shape,
                                transform=src.transform, fill=0).astype(bool)

metrics = {
    "Hillslope gradient (m/m)": read_raster(glob.glob("tweed_*polyfit_slope.tif")[0]),
    "Mean curvature (1/m)": read_raster(glob.glob("tweed_*polyfit_mean_curvature.tif")[0]),
    "Elevation (m)": read_raster("tweed_DEM.tif"),
}

pixels = pd.DataFrame({"unit": pd.Series(codes[inside]).map(unit_name)})
for name, raster in metrics.items():
    pixels[name] = raster[inside]
pixels = pixels.dropna()

area = (pixels.groupby("unit").size() * 30 * 30 / 1e6).rename("area_km2")
print(f"Catchment: {area.sum():.0f} km2")
summary = pixels.groupby("unit").median().join(area).sort_values("area_km2", ascending=False)
summary.round(3)

Now the charts: one panel per metric, the units in the same order and colours throughout. The first panel is the channel k<sub>sn</sub> of each node (from `tweed_ksn_chi_ksn_lithology_nodes_mn_*.csv`), the others are the pixels of each unit. Boxes span the 25th to 75th percentiles, with the median as a line; the whiskers reach 1.5 times the box height. The label under each box gives the unit's area in the catchment, or its channel length for k<sub>sn</sub>. Compare each unit's area and channel length when you compare its box with the others.

In [ ]:
nodes = pd.read_csv(f"tweed_ksn_chi_ksn_lithology_nodes_mn_{mn_suffix}.csv")
order = [u for u in summary.index if u in set(nodes.lith_name) | set(pixels.unit)]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
panels = [("k$_{sn}$ of channel nodes (m$^{" + f"{2 * best_mn:.2f}" + "}$)", None)] + [(m, m) for m in metrics]
for ax, (label, column) in zip(axes.ravel(), panels):
    if column is None:
        data = [nodes.k_sn[nodes.lith_name == u] for u in order]
        notes = [f"{nodes.flow_length_to_receiver_m[nodes.lith_name == u].sum() / 1000:.0f} km" for u in order]
    else:
        data = [pixels[column][pixels.unit == u] for u in order]
        notes = [f"{area.get(u, 0):.0f} km²" for u in order]
    keep = [i for i, d in enumerate(data) if len(d) > 0]
    box = ax.boxplot([data[i] for i in keep], showfliers=False, patch_artist=True,
                     medianprops={"color": "black"})
    for patch, i in zip(box["boxes"], keep):
        patch.set_facecolor(colours[order[i]])
        patch.set_edgecolor("black")
    ax.set_xticks(range(1, len(keep) + 1))
    ax.set_xticklabels([f"{order[i]}\n{notes[i]}" for i in keep], fontsize=8)
    ax.set_ylabel(label)
    ax.grid(axis="y", alpha=0.3)
axes[0, 0].set_title("Channels")
axes[0, 1].set_title("Hillslopes and valleys (all pixels)")
plt.tight_layout()
plt.show()

Finally, channel against hillslope: for each unit with at least 500 channel nodes, its median k<sub>sn</sub> against its median hillslope gradient.

**Question:** if one rock type were harder to erode than another, how would you expect these two medians to differ between them?

In [ ]:
pairs = summary.join(nodes.groupby("lith_name").k_sn.median().rename("k_sn_median"), how="inner")
pairs = pairs.join(nodes.groupby("lith_name").size().rename("n_nodes"))
pairs = pairs[pairs.n_nodes >= 500]          # units with at least 15 km of channel

fig, ax = plt.subplots(figsize=(6, 5))
for unit, row in pairs.iterrows():
    ax.scatter(row["Hillslope gradient (m/m)"], row["k_sn_median"], s=row["area_km2"] / 2 + 30,
               color=colours[unit], edgecolors="black", zorder=3)
    ax.annotate(unit, (row["Hillslope gradient (m/m)"], row["k_sn_median"]), xytext=(8, 4),
                textcoords="offset points")
ax.margins(0.3)
ax.set_xlabel("Median hillslope gradient (m/m)")
ax.set_ylabel("Median channel k$_{sn}$")
ax.set_title("Marker size: area of the unit in the catchment")
plt.show()

## Things to consider when comparing units

* **Where the units are.** A contrast between units is also a contrast between the parts of the catchment where they crop out. Compare the geological map with the network map and the elevation chart: do the units differ in elevation, in their position in the network, or in their distance from the outlet?
* **Nodes are not independent.** Each channel node carries the k<sub>sn</sub> of its fitted segment, so a unit's thousands of nodes may be a few dozen segments. The reach statistics and their bootstrap intervals (Step 4) take this into account.
* **The map is coarse.** At 1:625 000, small units and contacts are generalised. A channel along a contact can be in either unit.
* **The concavity is one value for the whole catchment.** k<sub>sn</sub> depends on m/n. With a different m/n from the indistinguishable range in Step 3, the k<sub>sn</sub> values change, and so may the contrasts between units.

## Things to try

* **Another m/n.** Set `best_mn` to the ends of the indistinguishable range from Step 3 and rerun Steps 4–6. Does the order of the units change?
* **A contact buffer.** Add `"lithology_contact_buffer_metres": 300` to `ksn_settings` to leave out channel nodes within 300 m of a contact, where the map may put them in the wrong unit.
* **Hillslopes only.** Leave out the valley floors in Step 6: for example, keep pixels with gradient above 0.1 or with positive mean curvature.
* **A different window.** Try `polyfit_lambda_metres: 60` or `150`. How sensitive are the gradients to the window?
* **Group the units.** A crosswalk file (`lithology_crosswalk_fname`, with columns `value,code,name`) can merge units, e.g. claystone, mudstone and `sedimentary` into one "fine sediments" unit. `lsdtt-fetch-lithology` writes a template, `tweed_crosswalk_template.csv`.

## Every run is recorded

Each program writes `<write_fname>_ingested_params.param`: every parameter it used. You can run it again as it is to repeat the run. Keep these files with your results.

## Data and references

* DEM: Copernicus GLO-30 (© DLR e.V. 2010–2014 and © Airbus Defence and Space GmbH 2014–2018, provided under COPERNICUS by the European Union and ESA).
* Locator map: drawn with GMT through PyGMT, from GMT's coastline, border and river data (GSHHG). Wessel, P., Luis, J. F., Uieda, L., Scharroo, R., Wobbe, F., Smith, W. H. F. and Tian, D. (2019). The Generic Mapping Tools version 6. *Geochemistry, Geophysics, Geosystems*, 20, 5556–5564. https://doi.org/10.1029/2019GC008515
* Geology: Macrostrat (https://macrostrat.org, CC BY 4.0), from the British Geological Survey 1:625 000 map (DiGMapGB-625, © NERC); see `tweed_ATTRIBUTION.txt`. Peters, S. E., Husson, J. M. and Czaplewski, J. (2018). Macrostrat: a platform for geological data integration and deep-time Earth crust research. *Geochemistry, Geophysics, Geosystems*, 19, 1393–1409. https://doi.org/10.1029/2018GC007467
* Perron, J. T. and Royden, L. (2013). An integral approach to bedrock river profile analysis. *Earth Surface Processes and Landforms*, 38(6), 570–576. https://doi.org/10.1002/esp.3302
* Mudd, S. M., Clubb, F. J., Gailleton, B. and Hurst, M. D. (2018). How concave are river channels? *Earth Surface Dynamics*, 6, 505–523. https://doi.org/10.5194/esurf-6-505-2018
* k<sub>sn</sub> by lithologic unit in lsdtt3: https://lsdtopotools.github.io/lsdtt3/design_and_algorithms/lithology_channel_statistics.html
* Every input of the programs used: https://lsdtopotools.github.io/lsdtt3/ (or run any program with `-h`).